In [ ]:
import numpy as np
import cabinetry
import pyhf
import json
import uproot
from pathlib import Path
import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt

cabinetry.set_logging()

# $B^+ \to K^+ (X(3872) \to J/\psi \pi^+ \pi^-)$ example 

Here we want to study the fitting of the $B^+ \to K^+ (X(3872) \to J/\psi \pi^+ \pi^-)$ channel, that you reconstructed in the workshop.
The [paper](https://arxiv.org/pdf/hep-ex/0309032) can be very helpful in this exercise.

There are some decisions to make before constructing a statistical model:

* **Fitting variable**: The choice of fitting variable is crucial. Ideally, one choses a variable where distributions of signal and background look very different. This gives more power to distinguish the two during the fitting procedure. Here we chose $\Delta E$, which is the difference between the energy of the $B$ and half the center of mass energy.

* **Binning**: The choice of binning is an important one. If the binning is too fine, we won't have enough statistics. If the binning is too wide, we lose information that might be important to us. 
In this example we chose a binning of 20 equally distributed bins in the range $-0.1 \geq \Delta E \geq 0.2$.

* **Samples**: We have to do a good job at modelling our data with MC samples. We include samples of different nature, to build a template. The benefit of keeping the samples separate is that we can modify them individually, and assign different modifiers to them. Here we have the following background samples:
  * **signal**: This includes the signal MC
  * **mixed**: This includes $B^0 \bar B^0$ background events.
  * **charged**: This includes $B^+ B^-$ background events.

# Environment setup

The data path is loaded from environment variables to make the notebook portable across different users and systems:

> **📝 Required Setup:**  
> **Create a `.env` file** in the project root with your paths:  
> `DATA_BASE_PATH=/your/path/to/data/`

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Load BASE path from environment variable
BASE = Path(os.getenv('DATA_BASE_PATH'))
    
print(f"Using data path: {BASE}")

# Building the model with `cabinetry`

We will use `cabinetry` ([documentation](https://cabinetry.readthedocs.io/en/latest/)) to build our `pyhf` model, providing many convenience functions to make our life easier. For example, `cabinetry` has the nice feature that it can crate `pyhf` models from `root` files directly. 

First we define the binning for our fitting variable.

In [ ]:
bins = np.linspace(-0.1, 0.2, 20 + 1)

First, let us load all the signal and background `ntuples`. We use a mix of reconstructed MC samples as the data in this example (charged and mixed $B \bar B$).

> **📝 Exercise:**   
> Follow the example to load all samples and plot a stacked histogram. How do shapes of signal and backgrounds vary? Do you think that $\Delta E$ is a good fitting variable? Why?

In [ ]:
with uproot.open({BASE / 'signal.root': 'tree'}) as tree:
    dat_signal = tree.arrays(tree.keys(), library="pd")

In [ ]:
plt.hist([dat_signal.B_deltaE], 
         bins=bins, 
         label=['Signal'], 
         stacked=True)
plt.xlabel('B_deltaE')
plt.ylabel('Counts')
plt.legend()
plt.show()

We define `cabinetry` models via a `config` dictionary, containing different settings - check out the [config documentation](https://cabinetry.readthedocs.io/en/latest/config.html).

We give our measurement a name, define a parameter of interest (POI), and input path containing the `root` files and a histogram folder, where cabinetry automatically saves the histogram yields.

In [ ]:
config = {
   'General':{
      'Measurement': 'B2KpipiJPsi',             # measurement name
      'POI': 'mu',                              # parameter of interest
      'InputPath': str(BASE / '{SamplePath}'),  # wildcard for samples
      'HistogramFolder': 'histograms/'          # output folder for histograms

   }
}

In the `Regions` setting, we tell `cabinetry` which variable in the `root` files it should load and define our signal region via the cut $ \Delta E > -0.1 GeV$ and $ \Delta E < 0.2 GeV$, applied through a `Filter` entry. This is strictly not needed here, but is a nice feature to see in action.

In [ ]:
global_cuts = '(B_deltaE>-0.1) & (B_deltaE<0.2)'               # global cuts applied to all regions

config.update({
   'Regions':[
      {
         'Name': 'signal_region',
         'Variable': 'B_deltaE',                               # which variable we bin histograms in
         'Binning': list(bins),
         'Filter': global_cuts
      }
   ]
})


Next we can define our `Samples` in a list, where we specify the name of each sample, the `root` file in the `InputPath`, the `Tree` and whether it is data or not. We can also pass a list of files as `SamplePath` and `cabinetry` will combine the files for us. Further, we can add weights to each sample, which can be a numeric scalar, applied to the full sample as a global weight, a column name in the `root` file for event-based weights, or a product of such. Also, to the samples can be filtered additionally.

If you do not have data points readily available, a good first check if your fit works properly is a fit to **Asimov data** (named after the writer Isaac Asimov). Asimov data is synthetic data, representing your exact expectations. It helps to validate your analysis before applying it to real data.

<a id="sample-config"></a>

> **📝 Exercise:**  
> Follow the provided code and add the backgrounds to the model.

> **📝 Exercise:**  
> Construct the `Data` samples, to represent an Asimov data-set. (`cabinetry` and `pyhf` can always give you the Asimov data-set for any model, here we pretend it is the real data.)

In [ ]:
config.update({
   'Samples':[
      {
         'Name': 'Data',
         'SamplePath': ['signal.root'],   # data files
         'Tree': 'tree',   # tree in root files
         'Data': True      # observed data is handled differently, need to distinguish
      },
      {
         'Name': 'signal',
         'SamplePath': 'signal.root',
         'Tree': 'tree',
         'Weight': '1.0'               # weight given to these samples (optional)
      },
   ]
})

Next, we can add some modifiers. First, we add some normalization factors for the signal. Here, we specify our `POI`, the signal normalization `mu`.

In [ ]:
config.update({
   'NormFactors':[               # type of modifiers that scale samples
      {
         'Name': 'mu',
         'Samples': 'signal',    # we want this parameter to scale the signal
         'Nominal': 1,
         'Bounds': [-10, 10],
      },
   ]
})

`cabinetry` lets us validate our `config`,


In [ ]:
cabinetry.configuration.validate(config)

Additionally, we can print an overview. We see that we have 5 samples, 1 region, 1 normalisation factor and no systematics so far. 

In [ ]:
cabinetry.configuration.print_overview(config)

## Creating the histograms

Given that our validation succeeds, we can `build` the histrograms for our model. This will create the hisrograms from the `root` files and save them into the `HistogramFolder`.

In [ ]:
cabinetry.templates.build(config, method='uproot')

You can also provide existing histograms you built yourself for `cabinetry` to use, see the [cabinetry-tutorials](https://github.com/cabinetry/cabinetry-tutorials) repository for an example.

`Cabinetry` also allows us to apply post-processing to our histograms, which consists of a fix for `NaN` statistical uncertainties and optional smoothing. It will create new histogram files in the `HistogramFolder` folder with the `*_modified.npz` ending.

In [ ]:
cabinetry.templates.postprocess(config)

We can now visualise what we produced:

In [ ]:
cabinetry.visualize.data_mc_from_histograms(config);

`cabinetry` will automatically save this image in a `/figures` folder.

# Adding background normalization systematics

We now want to make our model more realistic by adding systematic uncertainties.

We want to constrain the background normalizations, since we usually believe that our modelling of these is correct within a certain uncertainty. The tightness of these constraints is very case dependent. Here we chose a very conservative background normalization constraint of 50%.

> **📝 Exercise:**  
> Fill the name, you gave the samples above in the `Samples` entry

In [ ]:
norm_sys = [
      {
         'Name': 'norm_charged',
         'Samples': 'charged',            # Fill the name, you gave the samples above
         "Up": {"Normalization": 0.5},
         "Down": {"Normalization": -0.5},
         "Type": "Normalization"
      },
      {
         'Name': 'norm_mixed',
         'Samples': 'mixed',              # Fill the name, you gave the samples above
         "Up": {"Normalization": 0.5},
         "Down": {"Normalization": -0.5},
         "Type": "Normalization"
      },
   ]

config.update({'Systematics': norm_sys})
cabinetry.templates.build(config, method='uproot')

# Building a `pyhf` workspace

We now construct a `pyhf` workspace, which contains everything to build our likelihood function. This can also be used as an input file for `pyhf`. 

In [ ]:
workspace_path = 'workspace.json'
spec = cabinetry.workspace.build(config)

We can now print our workspace and inspect it.

In [ ]:
cabinetry.workspace.save(spec, workspace_path)
print(json.dumps(spec, sort_keys=True, indent=4))

## Model structure

It can be helpful to visualize the modifier structure of the statistical model we have built to catch potential issues. The `visualize.modifier_grid` function creates a figure showcasing the information about which modifiers (indicated by color) act on which region and sample when a given parameter (on the horizontal axis) is varied.

In [ ]:
cabinetry.visualize.modifier_grid(pyhf.Workspace(spec).model())

# Performing statistical inference

To perform inference, we need two things: a probability density function (pdf), or `model`, and data to fit it to. Both are derived from the workspace specification.

In [ ]:
model, data = cabinetry.model_utils.model_and_data(spec)

We see that all the modifiers that we defined for our model appear here.

Note that from this point onwards fits can easily be performed with [`pyhf.infer.mle.fit`](https://pyhf.readthedocs.io/en/v0.7.6/_generated/pyhf.infer.mle.fit.html#pyhf-infer-mle-fit) and you are encouraged to try.

## Maximum likelihood estimate

Let's fit our model to data to obtain the maximum likelihood estimate (MLE), trying to find the solution to the equation

$$\nabla_\lambda L(\vec n|\vec\lambda) = \vec 0.$$

In [ ]:
fit_results = cabinetry.fit.fit(model, data)

The fit converged, and we see the best-fit parameter results reported. The results are stored in a named tuple. This allows for easy access of the results. 

> **📝 Exercise:**  
> Do these fit results agree with your expectation? Especially if you set `Weights` in the [sample definition](#sample-config) above.

Note that background normalizations are calculated as $1 + {\rm norm} \cdot {\rm par}$, where we set ${\rm norm}=0.5$ in the uncertainty definition.

In [ ]:
for label, result, unc in zip(fit_results.labels, fit_results.bestfit, fit_results.uncertainty):
    print(f'{label}: {result:.3f} +/- {unc:.3f}')

### Post-fit yields

> **📝 Exercise:**  
> Use the above obtained `fit_results` to plot the post-fit yields using `cabinetry.model_utils.prediction` and `cabinetry.visualize.data_mc`.

> **📝 Bonus Question:**  
> How does cabinetry calculate the uncertainties on each bin?

### Pulls 

> **📝 Exercise:**  
> Plot the parameter pulls using `cabinetry.visualize.pulls`. What do you expect?

### Correlations 

Correlations are important to learn about how parameters ate co-dependent.

> **📝 Exercise:**  
> Use the `cabinetry.visualize.correlation_matrix` function to plot the correlation matrix. The parameter correlation matrix has a handy `pruning_threshold` setting to filter out parameters that are not highly correlated with others.

> **📝 Question:**  
> There is one parameter which is highly anti-correlated with the signal - why? Is this a problem?

### Likelihood scan

Likelihood scans will tell you about the (asymmetric) uncertainty of the parameter. You scan over the specified parameter and subsequently calculate the uncertainties using 

$$-2 \ln L(\hat \mu \pm \sigma_\mu) + 2 \ln L(\hat \mu) = 1$$

It will also tell you if your likelihood scan agrees well with a Gaussian approximation.

> **📝 Exercise:**  
> Perform a negative log likelihood scan over the parameter `mu` using `cabinetry.fit.scan`.

> **📝 Exercise:**  
> Visualize the scan results with `cabinetry.visualize.scan`.

> **📝 Question:**  
> Does it agree with the uncertainty obtained in the fit above? Does it agree well with a Gaussian approximation? Can you explain why?

# Template updates

> **📝 Exercise:**  
> To make the example more interesting, change the `Weight` of each sample [here](#sample-config) and see if you can recover the correct normalizations.

> **📝 Bonus Exercise:**  
> What do you expect the goodness-of-fit $P$-value to be? You can let `cabinetry` calculate this for you. Check the [documentation](https://cabinetry.readthedocs.io/en/latest/api.html#cabinetry.fit.fit).

# More advanced features

Here we provide a list of more advanced features, for you to study if you have time. Most features are documented in one of these referenced:

* [`Cabinetry` Documentation](https://cabinetry.readthedocs.io/en/latest/index.html)
* [`Cabinetry` tutorial](https://github.com/cabinetry/cabinetry-tutorials/blob/master/example.ipynb)
* [Tutorial from Belle II `pyhf` workshop](https://github.com/alexander-held/Belle-II-cabinetry/blob/main/talk.ipynb)